# Set Up

In [1]:
#Import variables
import os
exec(open("./PGS_calc_param.txt").read())

#Import packages
from datetime import datetime
import psutil
import time
import os
import subprocess

#Get CPU number
logical_cpus = psutil.cpu_count(logical=True)
print(logical_cpus)

64


# Write Scripts

In [10]:
%%writefile /home/jupyter/workspace/workspace-bucket/calculate_pgs/plink_bed_parallel.sh
#!/bin/bash

set -o pipefail
set -o errexit

echo "=============================="
echo "PLINK JOB START"
echo "PGS=${PGS}"
echo "BUILD=${BUILD}"
echo "CHROMS=${CHROMS}"
echo "START=$(date)"
echo "=============================="

SECONDS=0

INPUT_PATH="/home/jupyter/workspace/vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed"
OUTPUT_BASE="/home/jupyter/workspace/workspace-bucket/calculate_pgs/plink_results"

MAX_JOBS="${MAX_JOBS:-1}"

RUN_ID="$(date +%Y%m%d_%H%M%S)"

echo "MAX_JOBS=${MAX_JOBS}"


run_chr () {
    chrom=$1

    chr_start=$SECONDS
    echo "Starting chr${chrom}"

    bed_prefix="${INPUT_PATH}/chr${chrom}"

    memlog="${OUTPUT_BASE}/mem_logs/${RUN_ID}/mem_chr${chrom}.log"
    mkdir -p "$(dirname "${memlog}")"

    # memory monitor (isolated per job)
    (
        while true; do
        date
        grep -E "MemAvailable|MemTotal" /proc/meminfo
        sleep 30
        done
    ) > "${memlog}" &
    MEM_PID=$!
    trap 'kill "$MEM_PID" 2>/dev/null || true; wait "$MEM_PID" 2>/dev/null || true' EXIT

    read -r -a PGS_IDS <<< "${PGS}"

    PGS_TAG="${PGS// /_}"
    OUTPUT_PATH="${OUTPUT_BASE}/${PGS_TAG}_${BUILD}_${RUN_ID}"
    mkdir -p "${OUTPUT_PATH}" "${OUTPUT_BASE}/mem_logs/${RUN_ID}"

    RUN_INFO_FILE="${OUTPUT_BASE}/latest_run.env"
        {
            echo "RUN_ID=${RUN_ID}"
            echo "OUTPUT_PATH=${OUTPUT_PATH}"
            printf 'PGS=%q\n' "${PGS}"
            echo "BUILD=${BUILD}"
        } > "${RUN_INFO_FILE}"

    memlog="${OUTPUT_BASE}/mem_logs/${RUN_ID}/mem_chr${chrom}.log"
    score_list_file="${OUTPUT_PATH}/score_list_chr${chrom}.txt"
    : > "${score_list_file}"

    for PGS_ID in "${PGS_IDS[@]}"; do
        score_subset="${OUTPUT_PATH}/${PGS_ID}_chr${chrom}_score.txt"

        awk -F '\t' -v pgs="${PGS_ID}" -v chr="chr${chrom}:" \
            'BEGIN { OFS = "\t" }
             NR > 1 && $1 == pgs && index($2, chr) == 1 {
                 print $2, $3, $4
             }' \
            "${WEIGHTS_FILE}" > "${score_subset}"

        if [[ -s "${score_subset}" ]]; then
            printf '%s\n' "${score_subset}" >> "${score_list_file}"
        else
            echo "No variants for ${PGS_ID} on chr${chrom}; skipping score file."
            rm -f "${score_subset}"
        fi

    done

    if [[ -s "${score_list_file}" ]]; then
        plink2 \
            --bfile "${bed_prefix}" \
            --score-list "${score_list_file}" 1 2 3 \
            --threads "${PLINK_THREADS:-1}" \
            --out "${OUTPUT_PATH}/scores_chr${chrom}" \
            > "${OUTPUT_PATH}/plink_chr${chrom}.log" 2>&1
    else
        echo "No score files to run for chromosome ${chrom}."
    fi
    
    chr_elapsed=$((SECONDS - chr_start))
    printf "chr%s done in %02d:%02d:%02d\n" \
        "${chrom}" \
        $((chr_elapsed / 3600)) \
        $(((chr_elapsed % 3600) / 60)) \
        $((chr_elapsed % 60))
}

export -f run_chr
export INPUT_PATH OUTPUT_PATH WEIGHTS_PATH PGS BUILD

# -----------------------------
# PARALLEL EXECUTION CONTROL
# -----------------------------

for chrom in ${CHROMS}; do
    run_chr "${chrom}" &

    # enforce max parallel jobs
    while [[ $(jobs -r -p | wc -l) -ge ${MAX_JOBS} ]]; do
        sleep 2
    done
done

wait

elapsed=$SECONDS

echo "Finished at $(date)"
printf "All chromosome jobs complete in %02d:%02d:%02d\n" \
    $((elapsed / 3600)) \
    $(((elapsed % 3600) / 60)) \
    $((elapsed % 60))

Overwriting /home/jupyter/workspace/workspace-bucket/calculate_pgs/plink_bed_parallel.sh


# Execute

In [11]:
plink_start = time.time()

max_jobs = max(1, logical_cpus - 2)
plink_threads = max(1, max_jobs // 23)

#Use updated PGS_ID if available in param file, else use original PGS_ID
pgs_ids = globals().get("PGS_ID_updated", PGS_ID)

env = os.environ.copy()
env.update({
    "PGS": " ".join(pgs_ids),
    "BUILD": BUILD,
    "CHROMS": "1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 X",
    "WEIGHTS_FILE": COMBINED_WEIGHTS_FILE,
    "MAX_JOBS": str(max_jobs),
    "PLINK_THREADS": str(plink_threads)
})

print("MAX_JOBS:", env["MAX_JOBS"])

script = "/home/jupyter/workspace/workspace-bucket/calculate_pgs/plink_bed_parallel.sh"

subprocess.run(["chmod", "+x", script], check=True)
subprocess.run(["bash", script], check=True, env=env)


print(f"Done; Total time: {(time.time() - plink_start)/60:.1f} minutes")

MAX_JOBS: 62
PLINK JOB START
PGS=PGS000662 PGS000004 PGS003446 PGS002727 PGS002308 PGS002810
BUILD=GRCh38
CHROMS=1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 X
START=Tue Nov 10 03:26:00 PM UTC 2026
MAX_JOBS=62
Starting chr1
Starting chr2
Starting chr3
Starting chr4
Starting chr5
Starting chr6
Starting chr7
Starting chr8
Starting chr9
Starting chr10
Starting chr11
Starting chr12
Starting chr13
Starting chr14
Starting chr15
Starting chr16
Starting chr17
Starting chr18
Starting chr19
Starting chr20
Starting chr21
Starting chr22
Starting chrX


KeyboardInterrupt: 